<a href="https://colab.research.google.com/github/freeforall1932-design/dionysus/blob/arena/5bf33a0c-dionysus/workbench/projects/kobold-cli/kobold-cli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Kobold CLI — Dionysus coding/API edition

This notebook is adapted from the **official KoboldCpp Colab notebook**. The working upstream text-model download and launch mechanics are retained; the presets and defaults are changed for coding/API use. This edition includes text/coding models plus one Qwen3-VL image reader for screenshots/OCR; unrelated extra model sections have been removed.

**Default first run:** leave the quick preset on **Qwen3.5-9B Q5_K_M**, leave the manual loader checkboxes unchecked, and press **Play** on the big cell below. The first start still has to download KoboldCpp (about 0.65 GB) and the selected model (about 6.6 GB for this default), so it can take several minutes before the Cloudflare URL appears. Preset names in this notebook are just the model name plus quantization; the Hugging Face source, file size, preset context, and fit notes for every entry are in the project file [`MODEL_PRESETS.md`](https://github.com/freeforall1932-design/dionysus/blob/arena/5bf33a0c-dionysus/workbench/projects/kobold-cli/MODEL_PRESETS.md).

For API clients, use the printed tunnel URL plus `/v1` as the OpenAI-compatible base URL, and use the password printed by the cell as the Bearer token/API key. Do not share that password or save a copy of the notebook with cell outputs.

To choose manually instead of using a quick preset, check **LoadTextModel**, then select or paste a GGUF URL. For screenshots/OCR, the easiest path is to leave the manual checkboxes unchecked and choose the **Qwen3-VL-8B-Instruct Q6_K** quick preset; it loads the matched image projector automatically. You still need to upload/send the screenshot or image—the model cannot see your screen by itself.

Free-Colab memory is split, not pooled: a T4 is marketed as 16 GB VRAM but Colab commonly exposes about 15 GB usable GPU RAM, with roughly 12–13 GB separate system RAM on a standard runtime. For a strict full-GPU check, use usable VRAM − 1 GiB CUDA/driver − 1 GiB other runtime/compute − at least 1 GiB KV/context (more for longer context); at about 15 GiB usable, that leaves only about 12 GiB for model weights before any longer-context KV above 1 GiB. The Bonsai entries use only standard KoboldCpp-supported Bonsai packs (`Q1_0` and `Q2_0_g64`) and pass that strict screen with slack; earlier 13.x GB presets are tight under this stricter arithmetic and may need manual **Layers = Auto** and/or a lower context if a runtime reports OOM. Bonsai 2 special `PQ2_0`/`PTQ1_0` packs are deliberately not preset here because the checked KoboldCpp build does not yet contain Bonsai 2 runtime support. Manual **Layers = Auto** is also the slower spill-some-layers-to-system-RAM path. Actual allocations can vary, so check Colab's resource monitor for your runtime.

`DeleteExistingModels` now defaults **OFF**, so a model already downloaded in `/content` during this runtime is reused instead of being deleted and downloaded again. If you try several large presets in one runtime, disconnect/delete the runtime (or deliberately turn `DeleteExistingModels` ON before restarting) so `/content` does not fill with old model files.

The upstream silent-audio keep-alive cell is intentionally omitted; it is not part of model loading. When finished, stop the big cell and use **Runtime → Disconnect and delete runtime**.

Colab terms/session limits are still the project's open W-021/Q14 check; this owner-directed test notebook is not proof that a usage pattern is permitted.


In [ ]:
# @title <b>v-- Choose coding/API settings, then click Play to start KoboldCpp</b> {"display-mode":"form"}
#@markdown You will need to restart KoboldCpp with the play button when you make changes to these settings.
#@markdown Dionysus edition: upstream text-model download/load/launch mechanics are retained; unrelated extra model sections are removed.
# Adapted from LostRuins/koboldcpp colab.ipynb at concedo@f41f621e09822cc6ba48f3d791c41c239ebb6335 (git blob e47b699caed03db066575af01006d83067564b59); upstream license AGPL-3.0.

#@markdown ### Select a quick preset template (optional)
Template = "Qwen3.5-9B Q5_K_M" # @param ["Qwen3.5-9B Q5_K_M", "Qwen3.5-4B Q4_K_M", "Qwen3.5-27B Q3_K_M", "Qwen3-8B Q5_K_M", "Qwen3-14B Q5_K_M", "Qwen2.5-Coder-7B Q5_K_M", "Qwen2.5-Coder-14B Q4_K_M", "Qwen3-Coder-30B-A3B UD-IQ3_XXS", "Devstral-Small-2-24B Q4_K_S", "Ministral-3-8B-Instruct-2512 Q4_K_M", "Ministral-3-14B-Instruct-2512 Q4_K_M", "Mistral-Small-3.2-24B Q4_K_S", "GPT-OSS-20B MXFP4", "Gemma-4-E4B-it Q4_K_M", "Qwen3-VL-8B-Instruct Q6_K", "GLM-4.7-Flash Q3_K_S", "Phi-4 Q4_K", "Bonsai-8B Q1_0", "Bonsai-27B Q1_0", "Ternary-Bonsai-27B Q2_g64"]


#@markdown ### Or, choose manually by <u>selecting checkboxes and dropdown</u> below (Overwrites above templates!)

#@markdown <hr>
LoadTextModel = False #@param {type:"boolean"}
Model = "https://huggingface.co/unsloth/Qwen3.5-9B-GGUF/resolve/main/Qwen3.5-9B-Q5_K_M.gguf" #@param ["https://huggingface.co/unsloth/Qwen3.5-9B-GGUF/resolve/main/Qwen3.5-9B-Q5_K_M.gguf", "https://huggingface.co/unsloth/Qwen3.5-4B-GGUF/resolve/main/Qwen3.5-4B-Q4_K_M.gguf", "https://huggingface.co/unsloth/Qwen3.5-27B-GGUF/resolve/main/Qwen3.5-27B-Q3_K_M.gguf", "https://huggingface.co/Qwen/Qwen3-8B-GGUF/resolve/main/Qwen3-8B-Q5_K_M.gguf", "https://huggingface.co/Qwen/Qwen3-14B-GGUF/resolve/main/Qwen3-14B-Q5_K_M.gguf", "https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct-GGUF/resolve/main/qwen2.5-coder-7b-instruct-q5_k_m.gguf", "https://huggingface.co/Qwen/Qwen2.5-Coder-14B-Instruct-GGUF/resolve/main/qwen2.5-coder-14b-instruct-q4_k_m.gguf", "https://huggingface.co/unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF/resolve/main/Qwen3-Coder-30B-A3B-Instruct-UD-IQ3_XXS.gguf", "https://huggingface.co/unsloth/Devstral-Small-2-24B-Instruct-2512-GGUF/resolve/main/Devstral-Small-2-24B-Instruct-2512-Q4_K_S.gguf", "https://huggingface.co/mistralai/Ministral-3-8B-Instruct-2512-GGUF/resolve/main/Ministral-3-8B-Instruct-2512-Q4_K_M.gguf", "https://huggingface.co/mistralai/Ministral-3-14B-Instruct-2512-GGUF/resolve/main/Ministral-3-14B-Instruct-2512-Q4_K_M.gguf", "https://huggingface.co/unsloth/Mistral-Small-3.2-24B-Instruct-2506-GGUF/resolve/main/Mistral-Small-3.2-24B-Instruct-2506-Q4_K_S.gguf", "https://huggingface.co/ggml-org/gpt-oss-20b-GGUF/resolve/main/gpt-oss-20b-MXFP4.gguf", "https://huggingface.co/unsloth/gemma-4-E4B-it-GGUF/resolve/main/gemma-4-E4B-it-Q4_K_M.gguf", "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-Q6_K.gguf", "https://huggingface.co/unsloth/GLM-4.7-Flash-GGUF/resolve/main/GLM-4.7-Flash-Q3_K_S.gguf", "https://huggingface.co/microsoft/phi-4-gguf/resolve/main/phi-4-Q4_K.gguf", "https://huggingface.co/prism-ml/Bonsai-8B-gguf/resolve/main/Bonsai-8B-Q1_0.gguf", "https://huggingface.co/prism-ml/Bonsai-27B-gguf/resolve/main/Bonsai-27B-Q1_0.gguf", "https://huggingface.co/prism-ml/Ternary-Bonsai-27B-gguf/resolve/main/Ternary-Bonsai-27B-Q2_g64.gguf"] {"allow-input":true}
MdCommand = "" #@markdown <br>
Layers = "Auto" #@param ["Auto","999"]{allow-input: true}
ContextSize = "16384" #@param ["8192", "10240", "16384", "24576", "32768", "49152", "65536"] {allow-input: true}

#@markdown <hr>
#@markdown ### Image reader / OCR (Qwen3-VL only)
#@markdown Easiest: choose the Qwen3-VL quick preset above and leave all manual checkboxes unchecked; it loads the matched model and projector together. For manual mode, check both LoadTextModel and LoadVisionMMProjector and select the Qwen3-VL model above. You still need to upload/send a screenshot or image; the model cannot see your screen automatically.
LoadVisionMMProjector = False #@param {type:"boolean"}
Mmproj = "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/mmproj-BF16.gguf" #@param ["https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/mmproj-BF16.gguf"]
VCommand = ""

#@markdown <hr>
#@markdown Deletes all models, forcing a fresh redownload when you restart colab.
DeleteExistingModels = False #@param {type:"boolean"}

#@markdown <hr>
#@markdown This enables saving stories directly to your google drive. You will have to grant permissions, and then you can access the saves from the "KoboldCpp Server Storage" option.
AllowSaveToGoogleDrive = False #@param {type:"boolean"}
SavGdriveCommand = ""

#@markdown <hr>
#@markdown Only select the following box if regular cloudflare tunnel fails to work. It will generate an inferior localtunnel tunnel, which you can use after entering a password.
MakeLocalTunnelFallback = False #@param {type:"boolean"}

#@markdown <hr>
#@markdown ### Dionysus coding/API settings
#@markdown These settings only add small fragments to the upstream launch line. Model downloading/loading below remains the official KoboldCpp cell logic.
KVCache = "q8_0" #@param ["f16", "bf16", "q8_0", "q5_1", "q4_0"]
FlashAttention = True #@param {type:"boolean"}
BatchSize = "512" #@param ["128", "256", "512", "1024", "2048", "4096"] {allow-input: true}
GenLimit = 0 #@param {type:"integer"}
ParallelRequests = "1" #@param ["1", "2", "4"] {allow-input: true}
UseMMQ = True #@param {type:"boolean"}
EnableWebSearch = False #@param {type:"boolean"}
UseRemoteTunnel = True #@param {type:"boolean"}
GenerateApiPassword = True #@param {type:"boolean"}
ExtraArgs = "" #@param {type:"string"}
KCommand = ""
FACommand = ""
BCommand = ""
GCommand = ""
PCommand = ""
CudaModeCommand = "--usecuda 0"
SearchCommand = ""
TunnelCommand = ""
ApiPassword = ""

CustomCtxSize = ""
CustomMmproj = ""
Customized = False
has_checked_box = any((LoadTextModel, LoadVisionMMProjector))
if not has_checked_box:
  Model = ""
  Mmproj = ""
  if Template == "Qwen3.5-9B Q5_K_M":
    Customized = True
    Model = "https://huggingface.co/unsloth/Qwen3.5-9B-GGUF/resolve/main/Qwen3.5-9B-Q5_K_M.gguf"
    CustomCtxSize = "32768"
  if Template == "Qwen3.5-4B Q4_K_M":
    Customized = True
    Model = "https://huggingface.co/unsloth/Qwen3.5-4B-GGUF/resolve/main/Qwen3.5-4B-Q4_K_M.gguf"
    CustomCtxSize = "32768"
  if Template == "Qwen3.5-27B Q3_K_M":
    Customized = True
    Model = "https://huggingface.co/unsloth/Qwen3.5-27B-GGUF/resolve/main/Qwen3.5-27B-Q3_K_M.gguf"
    CustomCtxSize = "16384"
  if Template == "Qwen3-8B Q5_K_M":
    Customized = True
    Model = "https://huggingface.co/Qwen/Qwen3-8B-GGUF/resolve/main/Qwen3-8B-Q5_K_M.gguf"
    CustomCtxSize = "32768"
  if Template == "Qwen3-14B Q5_K_M":
    Customized = True
    Model = "https://huggingface.co/Qwen/Qwen3-14B-GGUF/resolve/main/Qwen3-14B-Q5_K_M.gguf"
    CustomCtxSize = "32768"
  if Template == "Qwen2.5-Coder-7B Q5_K_M":
    Customized = True
    Model = "https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct-GGUF/resolve/main/qwen2.5-coder-7b-instruct-q5_k_m.gguf"
    CustomCtxSize = "32768"
  if Template == "Qwen2.5-Coder-14B Q4_K_M":
    Customized = True
    Model = "https://huggingface.co/Qwen/Qwen2.5-Coder-14B-Instruct-GGUF/resolve/main/qwen2.5-coder-14b-instruct-q4_k_m.gguf"
    CustomCtxSize = "32768"
  if Template == "Qwen3-Coder-30B-A3B UD-IQ3_XXS":
    Customized = True
    Model = "https://huggingface.co/unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF/resolve/main/Qwen3-Coder-30B-A3B-Instruct-UD-IQ3_XXS.gguf"
    CustomCtxSize = "8192"
  if Template == "Devstral-Small-2-24B Q4_K_S":
    Customized = True
    Model = "https://huggingface.co/unsloth/Devstral-Small-2-24B-Instruct-2512-GGUF/resolve/main/Devstral-Small-2-24B-Instruct-2512-Q4_K_S.gguf"
    CustomCtxSize = "8192"
  if Template == "Ministral-3-8B-Instruct-2512 Q4_K_M":
    Customized = True
    Model = "https://huggingface.co/mistralai/Ministral-3-8B-Instruct-2512-GGUF/resolve/main/Ministral-3-8B-Instruct-2512-Q4_K_M.gguf"
    CustomCtxSize = "32768"
  if Template == "Ministral-3-14B-Instruct-2512 Q4_K_M":
    Customized = True
    Model = "https://huggingface.co/mistralai/Ministral-3-14B-Instruct-2512-GGUF/resolve/main/Ministral-3-14B-Instruct-2512-Q4_K_M.gguf"
    CustomCtxSize = "32768"
  if Template == "Mistral-Small-3.2-24B Q4_K_S":
    Customized = True
    Model = "https://huggingface.co/unsloth/Mistral-Small-3.2-24B-Instruct-2506-GGUF/resolve/main/Mistral-Small-3.2-24B-Instruct-2506-Q4_K_S.gguf"
    CustomCtxSize = "10240"
  if Template == "GPT-OSS-20B MXFP4":
    Customized = True
    Model = "https://huggingface.co/ggml-org/gpt-oss-20b-GGUF/resolve/main/gpt-oss-20b-MXFP4.gguf"
    CustomCtxSize = "32768"
  if Template == "Gemma-4-E4B-it Q4_K_M":
    Customized = True
    Model = "https://huggingface.co/unsloth/gemma-4-E4B-it-GGUF/resolve/main/gemma-4-E4B-it-Q4_K_M.gguf"
    CustomCtxSize = "32768"
  if Template == "Qwen3-VL-8B-Instruct Q6_K":
    Customized = True
    Model = "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-Q6_K.gguf"
    CustomCtxSize = "32768"
    CustomMmproj = "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/mmproj-BF16.gguf"
  if Template == "GLM-4.7-Flash Q3_K_S":
    Customized = True
    Model = "https://huggingface.co/unsloth/GLM-4.7-Flash-GGUF/resolve/main/GLM-4.7-Flash-Q3_K_S.gguf"
    CustomCtxSize = "8192"
  if Template == "Phi-4 Q4_K":
    Customized = True
    Model = "https://huggingface.co/microsoft/phi-4-gguf/resolve/main/phi-4-Q4_K.gguf"
    CustomCtxSize = "16384"
  if Template == "Bonsai-8B Q1_0":
    Customized = True
    Model = "https://huggingface.co/prism-ml/Bonsai-8B-gguf/resolve/main/Bonsai-8B-Q1_0.gguf"
    CustomCtxSize = "32768"
  if Template == "Bonsai-27B Q1_0":
    Customized = True
    Model = "https://huggingface.co/prism-ml/Bonsai-27B-gguf/resolve/main/Bonsai-27B-Q1_0.gguf"
    CustomCtxSize = "32768"
  if Template == "Ternary-Bonsai-27B Q2_g64":
    Customized = True
    Model = "https://huggingface.co/prism-ml/Ternary-Bonsai-27B-gguf/resolve/main/Ternary-Bonsai-27B-Q2_g64.gguf"
    CustomCtxSize = "32768"

if not Customized or Template=="" or has_checked_box:
  print("== Manual CHECKBOX OVERRIDE was detected, using manual selections ==")
  print(f"LoadTextModel: {LoadTextModel}")
  print(f"LoadVisionMMProjector: {LoadVisionMMProjector}")
elif Customized:
  print("== No checkbox override selected, loading a quick preset template! ==")
  print(f"CHOSEN PRSET: {Template}")
  print("If you want to load your own model selection, MAKE SURE you check one of the loader checkboxes...")
  if Model:
    LoadTextModel = True
    Layers = 999
  if CustomMmproj!="":
    Mmproj = CustomMmproj
    LoadVisionMMProjector = True
  if CustomCtxSize!="":
    ContextSize = CustomCtxSize

if Layers=="Auto":
  Layers = -1

print("=== STARTING THE LOAD ===")
if LoadTextModel:
  print(f"[Text Model] {Model}")
if LoadVisionMMProjector:
  print(f"[Vision MMProj] {Mmproj}")
import os, glob
if not os.path.isfile("/opt/bin/nvidia-smi"):
  raise RuntimeError("⚠️Colab did not give you a GPU due to usage limits, this can take a few hours before they let you back in. Check out https://lite.koboldai.net for a free alternative (that does not provide an API link but can load KoboldAI saves and chat cards) or subscribe to Colab Pro for immediate access.⚠️")

if AllowSaveToGoogleDrive:
  print("Attempting to request access to save to your google drive...")
  try:
    from google.colab import drive
    import os, json
    drive.mount('/content/drive', force_remount=True)
    if not os.path.exists("/content/drive/MyDrive"):
      raise RuntimeError("Google Drive mount failed. Please grant permissions and try again.")
    kcppdir = '/content/drive/MyDrive/koboldcpp_data'
    os.makedirs(kcppdir, exist_ok=True)
    savedatapath = os.path.join(kcppdir, "koboldcpp_save_db.jsondb")
    if not os.path.exists(savedatapath):
        settings_data = {}
        with open(savedatapath, "w") as json_file:
            json.dump(settings_data, json_file, indent=4)
        print(f"Created new koboldcpp_save_db.jsondb at {savedatapath}")
    else:
        print(f"Loading saved data at {savedatapath}")
    SavGdriveCommand = f" --savedatafile {savedatapath}"
  except Exception as e:
    print(f"⚠️ Error: {e}")
    print("Please ensure you grant Google Drive permissions and try again.")

%cd /content
if Model and LoadTextModel:
  MdCommand = f"--model {Model}"
else:
  MdCommand = ""
if Mmproj and LoadVisionMMProjector:
  VCommand = f"--mmproj {Mmproj}"
else:
  VCommand = ""
KCommand = f"--quantkv {KVCache}"
FACommand = "" if FlashAttention else "--noflashattention"
BCommand = f"--blasbatchsize {BatchSize}"
GCommand = f"--genlimit {GenLimit}" if int(GenLimit) > 0 else ""
PCommand = f"--parallelrequests {ParallelRequests}"
CudaModeCommand = "--usecuda 0" if UseMMQ else "--usecuda 0 nommq"
SearchCommand = "--websearch" if EnableWebSearch else ""
TunnelCommand = "--remotetunnel" if UseRemoteTunnel else ""
if GenerateApiPassword:
  import secrets
  ApiPassword = secrets.token_urlsafe(32)
  os.environ["KCPP_PASSWORD"] = ApiPassword
  os.environ["KCPP_ADMINPASSWORD"] = ApiPassword
  print("=" * 72)
  print(f"Dionysus API password / Bearer token for this run: {ApiPassword}")
  print("OpenAI-compatible base URL will be the printed tunnel URL + /v1. Copy this password, then clear cell outputs after startup.")
  print("=" * 72)
else:
  ApiPassword = ""
  os.environ.pop("KCPP_PASSWORD", None)
  os.environ.pop("KCPP_ADMINPASSWORD", None)
  print("WARNING: GenerateApiPassword is OFF; like the upstream default, no generated password will be set.")
print(f"[Dionysus settings] {CudaModeCommand} {KCommand} {FACommand} {BCommand} {GCommand} {PCommand} {SearchCommand} {TunnelCommand} ExtraArgs={ExtraArgs}")

if DeleteExistingModels:
  print("Deleting all cached models to redownload...")
  patterns = ['*.gguf', '*.bin', '*.ggml']
  for pattern in patterns:
      for file_path in glob.glob(pattern):
          try:
              os.remove(file_path)
              print(f"Deleted: {file_path}")
          except Exception as e:
              print(f"Failed to delete {file_path}: {e}")


!echo Downloading KoboldCpp, please wait...
!wget -O dlfile.tmp https://github.com/LostRuins/koboldcpp/releases/latest/download/koboldcpp-linux-x64 && mv dlfile.tmp koboldcpp_linux || (echo "Primary failed, trying mirror..." && wget -O dlfile.tmp https://koboldai.org/cpplinuxcu12 && mv dlfile.tmp koboldcpp_linux)
!test -f koboldcpp_linux && echo Download Successful || echo Download Failed
!chmod +x ./koboldcpp_linux
!apt update
!apt install aria2 -y

if MakeLocalTunnelFallback:
  import urllib
  print("Trying to use LocalTunnel as a fallback tunnel (not so good)...")
  ltpw = urllib.request.urlopen('https://loca.lt/mytunnelpassword').read().decode('utf8').strip("\n")
  !nohup npx --yes localtunnel --port 5001 > lt.log 2>&1 &
  !sleep 8
  print("=================")
  print("(LocalTunnel Results)")
  !cat lt.log
  print(f"Please open the above link, and input the password '{ltpw}'\nYour KoboldCpp will start shortly...")
  print("=================")
  !sleep 10
!./koboldcpp_linux $MdCommand $CudaModeCommand --gpulayers $Layers --contextsize $ContextSize $KCommand $FACommand $BCommand $GCommand $PCommand $SearchCommand --quiet $TunnelCommand $VCommand $SavGdriveCommand $ExtraArgs
